# GTEx model building with MOFA-FLEX

💡 **Environment:** `clamp-analyses`  

# Libraries

In [16]:
library(here)

set.seed(1)

In [17]:
library(reticulate)

# Name of the env you expect users to have
env_name <- "clamp-analyses"

if (nzchar(Sys.getenv("RETICULATE_PYTHON"))) {
  message("Using RETICULATE_PYTHON = ", Sys.getenv("RETICULATE_PYTHON"))

} else {
  conda <- Sys.which("conda")

  if (nzchar(conda)) {
    cmd <- sprintf(
      '%s run -n %s python -c "import sys; print(sys.executable)"',
      shQuote(conda), shQuote(env_name)
    )
    py <- tryCatch(system(cmd, intern = TRUE), error = function(e) character(0))

    if (length(py) == 1 && nzchar(py) && file.exists(py)) {
      Sys.setenv(RETICULATE_PYTHON = py)
      message("Auto-set RETICULATE_PYTHON = ", py)
    } else {
      message("Could not resolve env python via conda. Falling back to Sys.which('python').")
      Sys.setenv(RETICULATE_PYTHON = Sys.which("python"))
    }

  } else {
    message("conda not found on PATH. Falling back to Sys.which('python').")
    Sys.setenv(RETICULATE_PYTHON = Sys.which("python"))
  }
}

py_config()


Using RETICULATE_PYTHON = /home/msubirana/miniconda3/envs/clamp-analyses/bin/python



python:         /home/msubirana/miniconda3/envs/clamp-analyses/bin/python
libpython:      /home/msubirana/miniconda3/envs/clamp-analyses/lib/libpython3.11.so
pythonhome:     /home/msubirana/miniconda3/envs/clamp-analyses:/home/msubirana/miniconda3/envs/clamp-analyses
version:        3.11.14 | packaged by conda-forge | (main, Oct 22 2025, 22:46:25) [GCC 14.3.0]
numpy:          /home/msubirana/miniconda3/envs/clamp-analyses/lib/python3.11/site-packages/numpy
numpy_version:  2.3.5

NOTE: Python version was forced by RETICULATE_PYTHON

In [18]:
mfl <- import("mofaflex", delay_load = FALSE)
ad  <- import("anndata",  delay_load = FALSE)
np  <- import("numpy",    delay_load = FALSE)
pd  <- import("pandas",   delay_load = FALSE)

# Input

In [19]:
gtex_rds <- here("output/gtex/df_gtex_fbm_filt.rds")
k_rds    <- here("output/gtex/CLAMP_K_gtex.rds")

stopifnot(file.exists(gtex_rds), file.exists(k_rds))

gtex_data <- readRDS(gtex_rds)   # genes x samples
K <- readRDS(k_rds)              # scalar

dim(gtex_data)
K

stopifnot(is.numeric(K), length(K) == 1)
stopifnot(!is.null(rownames(gtex_data)), !is.null(colnames(gtex_data)))

[1] 21613 17382

[1] 412

In [20]:
# AnnData samples x genes
X <- t(as.matrix(gtex_data))
X_np <- np$array(X, dtype = "float32")
adata <- ad$AnnData(X_np)

# preserve names
adata$obs_names <- pd$Index(colnames(gtex_data))  # samples
adata$var_names <- pd$Index(rownames(gtex_data))  # genes

adata$var_names <- adata$var_names$str$upper()

## GO:BP gene sets from MSigDB and build annotations mask

In [21]:
dbver <- "7.5.1"

bp_collection <- mfl$tl$msigdb_get_features(
category = "c5.go.bp",
dbver = dbver
)

bp_collection <- bp_collection$filter(
adata$var_names,
min_fraction = 0.10,
min_count = 20,
max_count = 500
)

bp_collection <- bp_collection$merge_similar(
metric = "jaccard",
similarity_threshold = 0.8,
iteratively = TRUE
)

# store binary mask (genes x programs) in varm["annotations"]
# MOFA-FLEX tutorial stores the transposed mask

adata$varm[["annotations"]] <- bp_collection$to_mask(adata$var_names$tolist())$T

In [22]:
bp_collection

<FeatureSets 'c5.go.bp.v7.5.1.symbols' with 3074 feature sets>

In [26]:
bp_collection$to_mask(adata$var_names$tolist())$T

NULL

In [25]:
adata$varm[["annotations"]]

ERROR: 'annotations'

# MOFA-FLEX

In [23]:
data_list <- dict(group_1 = dict(view_1 = adata))

In [24]:
model <- mfl$MOFAFLEX(
  data_list,
  mfl$DataOptions(
    scale_per_group = FALSE,          # already z-scored
    plot_data_overview = FALSE,
    remove_constant_features = TRUE,
    annotations_varm_key = "annotations"
  ),
  mfl$ModelOptions(
    n_factors = as.integer(K),
    weight_prior = "Laplace"
  ),
  mfl$TrainingOptions(
    seed = 2510302247L,
    max_epochs = 2000L,
  )
)

ERROR: reduce() of empty iterable with no initial value

In [ ]:
# extract factors Z (samples x K) then convert to B (K x samples)
Z_dict <- model$get_factors(return_type = "numpy", ordered = FALSE)
Z <- py_to_r(Z_dict[["group_1"]])       # samples x K

B <- t(Z)                                 # K x samples
rownames(B) <- paste0("LV", seq_len(nrow(B)))
colnames(B) <- colnames(gtex_data)

# write outputs
output_dir <- here("output/gtex/mofa")
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

write.csv(
  B,
  file = file.path(output_dir, "gtex_B.csv"),
  quote = FALSE
)

# save model
py_save_object(model, file.path(output_dir, "mofaflex_model.pkl"))